# SnowMart 出店戦略ハンズオン — Day 2
## AI で出店戦略を立てる（60分）

### 前回のあらすじ
Day 1 では、スノーマート（全国500店舗のコンビニチェーン）のデータ基盤を構築しました。  
データベース作成、CSVロード、Marketplace天気データ、Time Travel、Dynamic Tables、Streamlit可視化まで一気に進めました。

### 今日のミッション
構築したデータ基盤に **AI** を載せて、「**どこに出店すべきか？**」という問いに答えます。

1. Cortex AI Functions でレビュー分析（感情分析・要約・分類）
2. Cortex Search で社内ナレッジ検索（RAG）
3. Cortex Agent でデータ × 文書を横断するAIエージェント
4. Cortex Code でAI駆動のデータ分析

## Scene 1: 振り返り & 動作確認（5分）
Day 1 のデータが残っているか確認しましょう。

In [ ]:
-- ロールとウェアハウスの設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
USE SCHEMA SNOWMART_DB.SNOWMART_SCHEMA;

In [ ]:
-- データの確認
SELECT 'SNOWMART_STORES' AS TBL, COUNT(*) AS CNT FROM SNOWMART_STORES
UNION ALL SELECT 'DAILY_SALES', COUNT(*) FROM DAILY_SALES
UNION ALL SELECT 'CUSTOMER_REVIEWS', COUNT(*) FROM CUSTOMER_REVIEWS
UNION ALL SELECT 'STORE_SALES_ANALYSIS', COUNT(*) FROM STORE_SALES_ANALYSIS;

> **確認**: Dynamic Table（STORE_SALES_ANALYSIS）にもデータが入っていますか？  
> Day 1 で作成した Dynamic Table は Snowflake が自動でリフレッシュしてくれています。

## Scene 2: Cortex AI Functions — レビュー分析（15分）
500件の顧客レビューを全部読む？そんな時間はありません。  
**SQL一行で感情分析・要約・分類**。Python不要、APIキー不要です。

### Step 2-1: 感情分析（SENTIMENT）

In [ ]:
-- レビューの感情スコアを計算（-1: ネガティブ 〜 +1: ポジティブ）
SELECT
    REVIEW_ID,
    STORE_ID,
    RATING,
    REVIEW_TEXT,
    SNOWFLAKE.CORTEX.SENTIMENT(REVIEW_TEXT) AS SENTIMENT_SCORE
FROM CUSTOMER_REVIEWS
ORDER BY SENTIMENT_SCORE ASC
LIMIT 20;

In [ ]:
-- 店舗別の平均感情スコア（低い店舗に課題あり）
SELECT
    cr.STORE_ID,
    st.STORE_NAME,
    st.PREFECTURE,
    COUNT(*) AS REVIEW_COUNT,
    ROUND(AVG(cr.RATING), 1) AS AVG_RATING,
    ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(cr.REVIEW_TEXT)), 3) AS AVG_SENTIMENT
FROM CUSTOMER_REVIEWS cr
JOIN SNOWMART_STORES st ON cr.STORE_ID = st.STORE_ID
GROUP BY cr.STORE_ID, st.STORE_NAME, st.PREFECTURE
HAVING COUNT(*) >= 3
ORDER BY AVG_SENTIMENT ASC
LIMIT 10;

### Step 2-2: テキスト要約（SUMMARIZE）
ネガティブレビューをまとめて要約し、全体の傾向を把握します。

In [ ]:
-- ネガティブレビューをまとめて要約
SELECT SNOWFLAKE.CORTEX.SUMMARIZE(
    ARRAY_TO_STRING(
        ARRAY_AGG(REVIEW_TEXT) WITHIN GROUP (ORDER BY RATING ASC),
        '\n'
    )
) AS NEGATIVE_REVIEW_SUMMARY
FROM CUSTOMER_REVIEWS
WHERE RATING <= 2;

### Step 2-3: テキスト分類（CLASSIFY_TEXT）
レビューを自動でカテゴリ分類。手動のタグ付け作業は不要です。

In [ ]:
-- レビューをカテゴリに自動分類
SELECT
    REVIEW_ID,
    REVIEW_TEXT,
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        REVIEW_TEXT,
        ['接客・サービス', '品揃え・商品', '立地・アクセス', '店内環境・清潔さ', '価格']
    ) AS CATEGORY
FROM CUSTOMER_REVIEWS
LIMIT 20;

**ポイント**
- `SNOWFLAKE.CORTEX.SENTIMENT()` — SQL一行で感情分析。Python不要、APIキー不要
- すべてSnowflake内で処理。データが外部に出ません
- AWSだと Comprehend + Lambda + API Gateway の構成が必要になるところです

## Scene 3: Cortex Search — 社内ナレッジ検索（10分）
スノーマートには出店ガイドラインという社内文書があります。  
「駅前出店の条件は？」と聞いたら、文書から関連箇所を見つけて答えてくれる仕組み（RAG）を作りましょう。

### Step 3-1: ガイドライン文書をテーブルに格納

In [ ]:
-- ドキュメント格納用テーブル
CREATE OR REPLACE TABLE STORE_DOCUMENTS (
    DOC_ID VARCHAR(10),
    DOC_TITLE VARCHAR(200),
    DOC_SECTION VARCHAR(200),
    DOC_TEXT VARCHAR(5000)
);

In [ ]:
-- 出店ガイドラインの主要セクションを挿入
INSERT INTO STORE_DOCUMENTS VALUES
('D001', '出店ガイドライン', '立地選定基準 - 必須条件',
 '最寄り駅から徒歩5分以内、または主要道路沿いで車のアクセスが良いこと。商圏人口（半径500m）が5,000人以上であること。店舗面積として最低60平方メートルを確保できること。24時間営業が可能な立地条件であること。'),
('D002', '出店ガイドライン', '立地選定基準 - 推奨条件',
 '昼間人口比率が1.2以上のエリアは弁当・飲料カテゴリの売上が高い傾向にある。大学や専門学校の半径1km以内は若年層の集客が見込める。病院・公共施設の近隣は安定した集客が期待できる。住宅地の場合、世帯数が3,000以上のエリアを優先する。'),
('D003', '出店ガイドライン', '競合分析の指針',
 '低密度エリア（半径500m内に競合0〜1店）は出店優先度が高い。中密度エリア（競合2〜3店）は差別化戦略が必要。高密度エリア（競合4店以上）は原則として出店を見送る。ただし駅直結や大型商業施設内など圧倒的優位性がある場合は例外とする。'),
('D004', '出店ガイドライン', '回避条件',
 '半径300m以内にスノーマートの既存店舗がある場合は原則出店しない。半径500m以内に競合コンビニが5店舗以上ある場合は差別化要因が明確でない限り出店を見送る。過去3年以内にコンビニの閉店が2件以上あったエリアは要注意とする。'),
('D005', '出店ガイドライン', '売上目標の目安',
 '都心商業地は日販50万円以上を目標。都心オフィス街は日販45万円以上。郊外住宅地は日販30万円以上。郊外ロードサイドは日販35万円以上。月間売上が800万円を下回る場合、3ヶ月連続で改善計画を策定する。'),
('D006', '出店ガイドライン', '天候・季節要因への対応',
 '梅雨時期は来客数が平均10〜15%減少するが客単価は上昇する傾向。夏季は飲料・アイスの売上が年間平均の1.5倍。冬季はおでん・中華まん等のFF売上が増加。台風・大雪時は来客数が50%以上減少する場合あり。'),
('D007', '出店ガイドライン', '2024年度の戦略変更点',
 'EC連携として店舗受取サービスの需要増に対応し宅配ボックス設置スペースの確保を推奨。省人化としてセルフレジ導入を標準化、新規出店は原則セルフレジ2台以上を設置。フードロス削減のためAIによる需要予測システムの導入を全店で推進中。'),
('D008', '出店ガイドライン', '競合チェーン別の特徴',
 'セブン-イレブンはPB商品が強く品質重視の顧客層。ファミリーマートはファストフードが強く若年層に人気。ローソンはスイーツ・ヘルシー系に強く女性客比率が高い。上記チェーンが手薄なカテゴリで差別化を図ることが重要。');

### Step 3-2: Cortex Search Service の作成
テーブルに格納した文書に対して、自然言語でセマンティック検索ができるサービスを作成します。

In [ ]:
-- Cortex Search Service を作成
CREATE OR REPLACE CORTEX SEARCH SERVICE SNOWMART_DOC_SEARCH
    ON DOC_TEXT
    ATTRIBUTES DOC_TITLE, DOC_SECTION
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 hour'
    AS (
        SELECT
            DOC_ID,
            DOC_TITLE,
            DOC_SECTION,
            DOC_TEXT
        FROM STORE_DOCUMENTS
    );

### Step 3-3: 検索テスト
自然言語でガイドラインを検索してみましょう。  
> Cortex Search Service の初回インデックス構築には数分かかることがあります。  
> エラーが出たら少し待ってから再実行してください。

In [ ]:
-- 「駅前の出店条件」を検索
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWMART_DB.SNOWMART_SCHEMA.SNOWMART_DOC_SEARCH',
        '{
            "query": "駅前に出店する場合の条件は何ですか？",
            "columns": ["DOC_SECTION", "DOC_TEXT"],
            "limit": 3
        }'
    )
) AS SEARCH_RESULTS;

In [ ]:
-- 「競合が多いエリアでの戦略」を検索
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWMART_DB.SNOWMART_SCHEMA.SNOWMART_DOC_SEARCH',
        '{
            "query": "競合が多いエリアではどうすればいいですか？",
            "columns": ["DOC_SECTION", "DOC_TEXT"],
            "limit": 3
        }'
    )
) AS SEARCH_RESULTS;

**ポイント**
- 社内文書を入れるだけで、自然言語で検索できるRAG基盤が完成
- ベクトル検索を自動で行うので、Embeddingモデルの選定やインデックス管理が不要
- AWSだと Bedrock Knowledge Base + OpenSearch Serverless の構成に相当します

## Scene 4: Cortex Agent / Snowflake Intelligence（15分）
いよいよ最終兵器、AIエージェントの登場です。  
売上データの分析（Cortex Analyst）と社内文書の検索（Cortex Search）を統合し、  
自然言語で「どこに出店すべきか？」と聞けるエージェントを作ります。

### Step 4-1: Semantic View の作成（Cortex Analyst 用）
AIがSQLを自動生成するために、テーブルの「意味」を定義します。

In [ ]:
-- Cortex Analyst 用の Semantic View を作成
CREATE OR REPLACE SEMANTIC VIEW SNOWMART_DB.SNOWMART_SCHEMA.SNOWMART_SEMANTIC_VIEW
  AS SEMANTIC MODEL YAML $$
semantic_model:
  name: snowmart_analysis
  description: "スノーマートの店舗・売上・エリア分析用セマンティックモデル"

  tables:
    - name: store_sales_analysis
      base_table:
        database: SNOWMART_DB
        schema: SNOWMART_SCHEMA
        table: STORE_SALES_ANALYSIS
      dimensions:
        - name: store_id
          expr: STORE_ID
          description: "店舗ID"
        - name: store_name
          expr: STORE_NAME
          description: "店舗名"
        - name: prefecture
          expr: PREFECTURE
          description: "都道府県"
        - name: city
          expr: CITY
          description: "市区町村"
        - name: store_type
          expr: STORE_TYPE
          description: "店舗タイプ（直営 or FC）"
        - name: area_type
          expr: AREA_TYPE
          description: "エリアタイプ（商業地、住宅地、オフィス街、郊外、駅前繁華街）"
        - name: nearest_station
          expr: NEAREST_STATION
          description: "最寄り駅"
        - name: sales_date
          expr: SALES_DATE
          description: "売上日"
      measures:
        - name: total_sales
          expr: SUM(SALES_AMOUNT)
          description: "売上合計"
        - name: total_customers
          expr: SUM(CUSTOMER_COUNT)
          description: "来客数合計"
        - name: avg_unit_price
          expr: AVG(AVG_UNIT_PRICE)
          description: "平均客単価"
        - name: avg_daily_sales
          expr: AVG(SALES_AMOUNT)
          description: "平均日販"
        - name: total_food_sales
          expr: SUM(FOOD_SALES)
          description: "食品売上合計"
        - name: total_beverage_sales
          expr: SUM(BEVERAGE_SALES)
          description: "飲料売上合計"
        - name: store_count
          expr: COUNT(DISTINCT STORE_ID)
          description: "店舗数"
        - name: population
          expr: MAX(POPULATION)
          description: "エリア人口"
        - name: daytime_pop_ratio
          expr: MAX(DAYTIME_POPULATION_RATIO)
          description: "昼間人口比率"
        - name: avg_income
          expr: MAX(AVG_ANNUAL_INCOME)
          description: "平均年収"
        - name: floor_area
          expr: AVG(FLOOR_AREA_SQM)
          description: "平均売場面積"
$$;

### Step 4-2: Cortex Agent の作成
Semantic View（構造化データ）と Cortex Search（非構造化データ）を統合したAIエージェントを作成します。

In [ ]:
CREATE OR REPLACE CORTEX AGENT SNOWMART_AGENT
  COMMENT = 'スノーマート出店戦略AIエージェント'
FROM SPECIFICATION $$
models:
  orchestration: auto

instructions:
  system: |
    あなたはスノーマート（コンビニチェーン）の出店戦略アドバイザーです。
    売上データ、エリア情報、競合情報、社内ガイドラインを横断的に活用して、
    出店戦略に関する質問に答えてください。日本語で回答してください。
  orchestration: |
    売上や店舗の数値に関する質問は Analyst ツールを使ってください。
    出店基準や社内ルールに関する質問は Search ツールを使ってください。
    出店戦略の提案では、両方のツールを組み合わせて根拠のある回答をしてください。
  response: |
    常に具体的な数値やデータに基づいて回答してください。
    回答は簡潔に、箇条書きを活用してください。

tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: Analyst
      description: "売上データ、店舗情報、エリア情報を分析するツール"
  - tool_spec:
      type: cortex_search
      name: DocSearch
      description: "出店ガイドライン等の社内文書を検索するツール"
  - tool_spec:
      type: data_to_chart
      name: data_to_chart
      description: "データをチャートで可視化するツール"

tool_resources:
  Analyst:
    semantic_view: "SNOWMART_DB.SNOWMART_SCHEMA.SNOWMART_SEMANTIC_VIEW"
  DocSearch:
    name: "SNOWMART_DB.SNOWMART_SCHEMA.SNOWMART_DOC_SEARCH"
    max_results: "5"
$$;

### Step 4-3: エージェントと対話（Snowflake Intelligence）

#### Snowflake Intelligence を開く
1. Snowsight 左メニュー → **AI & ML** → **Cortex Agent**
2. **SNOWMART_AGENT** を選択

#### 以下の質問を試してみましょう

**質問 1（データ分析）**:  
「売上トップ10の店舗を教えてください。共通する特徴はありますか？」

**質問 2（都道府県比較）**:  
「都道府県別の平均日販を比較してください」

**質問 3（ガイドライン参照）**:  
「競合が3店以上あるエリアに出店する場合、何に注意すべきですか？」

**質問 4（出店提案 — 横断回答）**:  
「横浜市に新規出店するとしたら、どのエリアタイプが最適ですか？根拠も含めて教えてください」

> **注目**: エージェントが質問に応じて自動的にツール（Analyst / DocSearch / Chart）を選択・組み合わせている点を確認してください。

**ポイント**
- 構造化データ（SQL） × 非構造化データ（文書検索）を1つのエージェントが横断して回答
- Text-to-SQL + RAG + チャート生成がオールインワン
- AWSだと Bedrock Agent + Athena + OpenSearch + QuickSight を統合する必要があります

## Scene 5: Cortex Code でデータ分析（10分）
最後に、Cortex Code を使ったAI駆動のデータ分析を体験します。  
自然言語で指示するだけで、SQLを書いて実行し、結果を解釈してくれます。

### Cortex Code in Snowsight を開く
1. Snowsight の **SQL Worksheet** を開く
2. AI アシスタント（Cortex）を使って以下のプロンプトを試す

**プロンプト 1（探索的分析）**:  
「STORE_SALES_ANALYSIS テーブルを使って、直営店とFC店の売上パフォーマンスを比較するクエリを書いてください」

**プロンプト 2（異常値検出）**:  
「日販が極端に低い店舗を特定するクエリを書いてください。月別の推移も見たいです」

**プロンプト 3（出店候補分析）**:  
「COMPETITOR_STORES と AREA_MASTER を使って、競合が少なく人口が多いエリアを見つけるクエリを書いてください」

> **ヒント**: SQLの文法を覚えていなくても、やりたいことを日本語で伝えればOK。  
> 生成されたSQLを確認・修正してから実行できるので安全です。  
> GitHub Copilot の SQL版、というイメージです。

## Scene 6: まとめ & 次のステップ（5分）

### 2日間のまとめ

**Day 1 でやったこと：**
- データベース構築 & CSVロード
- Marketplace から外部データを即取得
- Time Travel & Clone で安全なデータ管理
- Warehouse サイズ変更でパフォーマンス調整
- Dynamic Tables で自動更新パイプライン
- Streamlit でダッシュボード構築（AIでコード改善）

**Day 2 でやったこと：**
- Cortex AI Functions でレビューの感情分析・要約・分類
- Cortex Search で社内文書の自然言語検索（RAG）
- Cortex Agent で全データソースを横断するAIエージェント
- Cortex Code でAI駆動のデータ分析

### まとめ
Snowflake は単なるデータベースではなく、**データ基盤からAI活用までをカバーするプラットフォーム**です。  
AWSのサービスでいえば、RDS + S3 + Glue + Athena + Data Exchange + SageMaker + Bedrock + QuickSight に相当する機能が、1つのプラットフォームに統合されています。  

そして今日使った全ての機能は、**データがSnowflakeの外に出ることなく動作**しています。  
ガバナンスとセキュリティが保たれたまま、AIを活用できる。これがSnowflakeの強みです。

### 参考リンク
- [Snowflake 公式ドキュメント](https://docs.snowflake.com/)
- [Cortex AI Functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/llm-functions)
- [Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents)
- [Snowflake Marketplace](https://app.snowflake.com/marketplace)
- [Cortex Code CLI](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code)

## 付録: クリーンアップ
ハンズオン終了後にリソースを削除する場合は以下を実行してください。

In [ ]:
-- 作成したオブジェクトを削除
DROP DATABASE IF EXISTS SNOWMART_DB;
DROP DATABASE IF EXISTS SNOWMART_DB_DEV;

-- ウェアハウスのサイズを戻す
ALTER WAREHOUSE COMPUTE_WH SET WAREHOUSE_SIZE = 'XSMALL';

## 付録: トラブルシューティング

**Q: Cortex AI Functions がエラーになる**  
→ `SNOWFLAKE.CORTEX` スキーマへのアクセス権を確認。Enterprise Edition 以上が必要。

**Q: Dynamic Table のデータが空**  
→ `SHOW DYNAMIC TABLES` でステータスを確認。リフレッシュ完了まで数分待つ。

**Q: Cortex Search Service の作成に時間がかかる**  
→ 初回のインデックス構築には数分かかることがあります。`SHOW CORTEX SEARCH SERVICES` で確認。

**Q: Day 2 でデータが消えている**  
→ トライアルアカウントの有効期限（30日）を確認。  
→ ウェアハウスが SUSPENDED なら `ALTER WAREHOUSE COMPUTE_WH RESUME` を実行。